# IndicCMix — native-script + Latin-English candidate collection

Purpose: collect candidate rows from `ai4bharat/IndicCMix` where the utterance uses the target language's native script and English in Latin script in the SAME text. These are **synthetic/translation-derived candidates**, not proof of natural voice-call distribution. Keep them candidate-only and do not use them as independent evaluation data.

Dataset card reports MIT license. Re-check current license and provenance before commercial use. The audio in the dataset is English TTS and is not aligned as spoken code-mixed speech; this notebook uses text only.

Source: https://huggingface.co/datasets/ai4bharat/IndicCMix

In [ ]:
!pip -q install datasets pandas


## Authenticate with Hugging Face (required)

First open https://huggingface.co/datasets/ai4bharat/IndicCMix in a browser, sign in, and accept/request access if prompted. Then run the cell below and paste your **read** token into the private prompt. Do not hardcode the token in the notebook or commit it to GitHub.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from datasets import load_dataset
from pathlib import Path
import pandas as pd, re, hashlib, json

OUT = Path('/content/data/external/indiccmix')
OUT.mkdir(parents=True, exist_ok=True)
TARGET_PER_CATEGORY = 20000

# Language file codes from the dataset card (Kannada uses 'ka').
LANGS = {
 'hi': {'category':'Hindi-English Hinglish mixed-script','label':'hi-IN','native':r'[\u0900-\u097F]'},
 'pa': {'category':'Punjabi-English mixed-script','label':'pa-IN','native':r'[\u0A00-\u0A7F]'},
 'bn': {'category':'Bengali-English mixed-script','label':'bn-IN','native':r'[\u0980-\u09FF]'},
 'gu': {'category':'Gujarati-English mixed-script','label':'gu-IN','native':r'[\u0A80-\u0AFF]'},
 'ka': {'category':'Kannada-English mixed-script','label':'kn-IN','native':r'[\u0C80-\u0CFF]'},
 'ml': {'category':'Malayalam-English mixed-script','label':'ml-IN','native':r'[\u0D00-\u0D7F]'},
 'mr': {'category':'Marathi-English mixed-script','label':'mr-IN','native':r'[\u0900-\u097F]'},
 'te': {'category':'Telugu-English mixed-script','label':'te-IN','native':r'[\u0C00-\u0C7F]'},
 'as': {'category':'Assamese-English mixed-script','label':'as-IN','native':r'[\u0980-\u09FF]'},
 'ta': {'category':'Tamil-English mixed-script','label':'ta-IN','native':r'[\u0B80-\u0BFF]'},
 'or': {'category':'Unsupported-language-English','label':'none','native':r'[\u0B00-\u0B7F]'},
}
LATIN = re.compile(r'[A-Za-z]{2,}')


## Collect text candidates (no audio download)

The two-script gate is automated, but it cannot prove that Latin tokens are truly English or that the Indic text is linguistically correct. Inspect samples and manually review before approval. English words kept in Latin are a property of this dataset's construction; do not mistake this for natural user-generated call data.

In [ ]:
summaries=[]
for lang, cfg in LANGS.items():
    print('Loading language:', lang, cfg['category'])
    # IndicCMix card lists language files as <code>.parquet; Kannada uses 'ka'.
    ds = load_dataset('ai4bharat/IndicCMix', data_files=f'{lang}.parquet', split='train', token=True)
    rows=[]; seen=set(); scanned=0
    for item in ds:
        scanned += 1
        txt = str(item.get('native_script_codemixed','')).strip()
        if not txt: continue
        if not re.search(cfg['native'], txt) or not LATIN.search(txt): continue
        norm = re.sub(r'\s+',' ',txt.casefold()).strip()
        if norm in seen: continue
        seen.add(norm)
        source_id = str(item.get('id', f'{lang}_{scanned}'))
        # Use the source sentence id to keep aligned language variants together during splitting.
        group_id = hashlib.sha256(('indiccmix|' + source_id.split('_')[1] if '_' in source_id else 'indiccmix|' + source_id).encode()).hexdigest()[:20]
        rows.append({'text':txt,'label':cfg['label'],'language_category':cfg['category'],'source_dataset':'AI4Bharat IndicCMix','source_url':'https://huggingface.co/datasets/ai4bharat/IndicCMix','source_row_id':source_id,'group_id':group_id,'source_split':'train','is_synthetic':True,'review_status':'pending_language_quality_and_domain_review','license_status':'MIT_reported_verify_current','contains_native_script':True,'contains_latin_english_candidate':True})
        if len(rows) >= TARGET_PER_CATEGORY: break
    path=OUT/(lang+'_mixed_script_candidates.csv')
    pd.DataFrame(rows).to_csv(path,index=False)
    summaries.append({'language_code':lang,'category':cfg['category'],'label':cfg['label'],'rows_collected':len(rows),'rows_scanned':scanned,'target':TARGET_PER_CATEGORY,'status':'candidate_only_not_approved','path':str(path)})
    print('Saved',len(rows),'rows; scanned',scanned)
summary=pd.DataFrame(summaries)
display(summary)
summary.to_csv(OUT/'indiccmix_collection_summary.csv',index=False)


In [ ]:
from google.colab import files
import zipfile
zip_path=OUT/'indiccmix_candidates.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*_candidates.csv'): z.write(p,arcname=p.name)
files.download(str(OUT/'indiccmix_collection_summary.csv'))
files.download(str(zip_path))
